In [0]:
from pyspark.sql import functions as F

def get_trip_event_rules():
    return [
        {
            "name": "trip_id_not_null",
            "condition": F.expr("trip_id IS NOT NULL"),
            "description": "trip_id must not be null"
        },
        {
            "name": "event_id_not_null",
            "condition": F.expr("event_id IS NOT NULL"),
            "description": "event_id must not be null"
        },
        {
            "name": "driver_required_for_started_trip",
            "condition": F.expr("""
                event_type NOT IN ('TRIP_STARTED','TRIP_COMPLETED','PAYMENT_COMPLETED','DRIVER_ASSIGNED','TRIP_CANCELLED')
                OR driver_id IS NOT NULL
            """),
            "description": "driver_id must be present once a driver has been assigned to the trip"
        },
        {
            "name": "positive_distance",
            "condition": F.expr("distance_km IS NULL OR distance_km > 0"),
            "description": "distance_km must be positive when present"
        },
        {
            "name": "non_negative_fare",
            "condition": F.expr("fare_amount IS NULL OR fare_amount >= 0"),
            "description": "fare_amount must not be negative"
        },
        {
            "name": "valid_pickup_zone",
            "condition": F.expr("pickup_zone_id IS NULL OR pickup_zone_id != 9999"),
            "description": "pickup_zone_id must reference a known zone, not the unknown-zone sentinel"
        },
        {
            "name": "valid_dropoff_zone",
            "condition": F.expr("dropoff_zone_id IS NULL OR dropoff_zone_id != 9999"),
            "description": "dropoff_zone_id must reference a known zone"
        },
        {
            "name": "dropoff_after_pickup",
            "condition": F.expr("""
                dropoff_datetime IS NULL OR pickup_datetime IS NULL
                OR dropoff_datetime >= pickup_datetime
            """),
            "description": "dropoff_datetime must not be earlier than pickup_datetime"
        },
        {
            "name": "surge_multiplier_valid",
            "condition": F.expr("surge_multiplier IS NULL OR (surge_multiplier >= 1.0 AND surge_multiplier <= 5.0)"),
            "description": "surge_multiplier must be between 1.0 and 5.0"
        },
    ]